[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L03.ipynb)

# Lecture 03 — Your First Complete Model

*See the whole pipeline run once — data, model, loss, optimizer, loop — before we explain every piece.*

**PyTorch from Ground Up · Volume I · Chapter 3** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Name the five parts every PyTorch project shares.
- Build and train a one-layer model that discovers y = 3x + 1 from noisy data.
- Recite the five-line training loop and say what each line does.
- Read a loss curve: falling, flat or rising.
- Explain what happens when `zero_grad()` is forgotten or the learning rate is wrong.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · A slightly reckless plan: the five-part pipeline

Today we train a complete model **before** learning the vocabulary. Don’t try to understand every line — watch the whole thing run, from raw numbers to a model that learns, so every piece we study later has a place to land. Think of it as the trailer for the rest of the course.

<sub>Source: lecture slides 3–5</sub>

## 02 · Step 1 — the data

Twenty noisy points — the noise forces the model to find the **trend**, not memorize. x and y are both shape `(20, 1)`. The `0.3 * randn_like(x)` term scatters the points around the true line. That is the difference between lookup and learning.

<sub>Source: lecture slides 6–7</sub>

**`the_data.py`** — predict the output, then run.

In [ ]:
import torch
torch.manual_seed(0)
# 20 training points, x from 0 to 2
x = torch.linspace(0, 2, 20).unsqueeze(1)   # (20, 1)
y = 3 * x + 1 + 0.3 * torch.randn_like(x)   # noisy
print(x.shape, y.shape)

<details><summary>▸ Expected output</summary>

```
torch.Size([20, 1]) torch.Size([20, 1])
```
</details>

## 03 · Step 2 — the model

A linear layer is just **multiply and add**: a weight (slope) and a bias (intercept). Both start as small random numbers, so today’s predictions are random. Our whole job is to nudge those two numbers until the predictions match the data.

<sub>Source: lecture slides 8–10</sub>

**`the_model.py`** — predict the output, then run.

In [ ]:
import torch.nn as nn
model = nn.Linear(1, 1)          # 1 input, 1 output
print(model.weight, model.bias)

<details><summary>▸ Expected output</summary>

```
Two Parameter tensors, filled with small random numbers.
```
</details>

> ⚠️ **Watch out — Shape discipline starts here**  
> The model expects input of shape `(batch, features)`. That’s exactly why we made x shape `(20, 1)` — 20 examples, 1 feature each — rather than a flat `(20,)`. Get this wrong and the layer either errors or silently does the wrong thing.
>
> Shape discipline starts here and never stops.

## 04 · Steps 3 & 4 — the loss and the optimizer

Training needs one number that answers: **how wrong are we right now?** For a continuous output that number is **mean squared error**: take each prediction’s error, square it, average. Zero means perfect — and training exists to drive it down.

<sub>Source: lecture slides 11–14</sub>

**`the_loss.py`** — predict the output, then run.

In [ ]:
loss_fn = nn.MSELoss()

**`the_optimizer.py`** — predict the output, then run.

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1)

## 05 · Step 5 — the training loop

Loss 20.3 → 0.07 in 200 tiny steps. **That is learning.** Now name the five lines — you will write them, in this order, in almost every project you ever build.

<sub>Source: lecture slides 15–16</sub>

**`training_loop.py`** — predict the output, then run.

In [ ]:
for epoch in range(200):
    pred = model(x)              # ❶ forward pass
    loss = loss_fn(pred, y)      # ❷ measure error
    optimizer.zero_grad()        # ❸ reset gradients
    loss.backward()              # ❹ compute gradients
    optimizer.step()             # ❺ update parameters
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} Loss: {loss.item():.4f}")

<details><summary>▸ Expected output</summary>

```
Epoch   0 Loss: 20.2505
Epoch  50 Loss: 0.1018
Epoch 100 Loss: 0.0678
Epoch 150 Loss: 0.0667
```
</details>

## 06 · Did it learn? Watch it happen

3.29 and 0.82 — **discovered from noisy points**, never told the formula. Not exactly 3 and 1 because twenty noisy points genuinely lean that way: 3.29x + 0.82 is the best possible line through this data. Real models chase a signal buried in noise — close is the win.

<sub>Source: lecture slides 17–19, 22–23</sub>

**`check_answer.py`** — predict the output, then run.

In [ ]:
w = model.weight.item()
b = model.bias.item()
print(f"Learned: y = {w:.2f}x + {b:.2f}")
print(f"Truth:   y = 3.00x + 1.00")

<details><summary>▸ Expected output</summary>

```
Learned: y = 3.29x + 0.82
Truth:   y = 3.00x + 1.00
```
</details>

🧪 *Interactive lab in the app:* **Your first complete model — live** — try it there to see this idea move.

> ⚠️ **Watch out — Never forget zero_grad**  
> If you drop `optimizer.zero_grad()`, the gradients from every previous step pile up instead of resetting, and the model quickly diverges — the loss blows up instead of falling.
>
> It is the most common single-line bug in PyTorch training loops. Tick the box in the lab above and see it for yourself.

## 07 · Everything together

Fifteen lines. The entire machine-learning pipeline, in miniature. Run it yourself. The rest of the course takes this apart piece by piece — and puts it back together with more and more power.

<sub>Source: lecture slides 20–21, 24</sub>

**`complete_model.py`** — predict the output, then run.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

# --- Data ---
x = torch.linspace(0, 2, 20).unsqueeze(1)
y = 3 * x + 1 + 0.3 * torch.randn_like(x)

# --- Model, Loss, Optimizer ---
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1)

# --- Training loop ---
for epoch in range(200):
    loss = loss_fn(model(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

w = model.weight.item()
b = model.bias.item()
print(f"y ≈ {w:.2f}x + {b:.2f}")

<details><summary>▸ Expected output</summary>

```
y ≈ 3.29x + 0.82
```
</details>

## The rule — worth remembering

**What to carry from today**

1. Every project follows the same five steps: Data → Model → Loss → Optimizer → Loop.
2. The training loop has five lines: forward, loss, `zero_grad`, backward, step.
3. The loss curve is the proof: falling means learning.
4. You do not need to understand every piece today. Seeing the whole once is enough.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Run it, then change it

Run the complete listing. Then change the true relationship to `y = 5x + 2` (modify only the data line) and train again. Does the model find the new slope and intercept?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes — it lands at y ≈ 5.29x + 1.82: near 5 and 2, off by the same noise as before. Nothing about the model, loss, optimizer or loop changed: the same skeleton learns whatever line the data describes. Try it in the live lab.
</details>

### Exercise 02 · 01 · A different line

Change the data to `y = -2x + 5` (with noise) and retrain — what does the model learn?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

y ≈ −1.71x + 4.81 — a weight near −2 and a bias near 5. Negative slopes are no harder than positive ones.
</details>

### Exercise 03 · 02 · Learning-rate sweep

Change the learning rate to 0.001, then to 10.0; run 200 epochs each. How does the final loss change, and what happens at 10.0? (Hint: watch for `nan`.)

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

At `lr=0.001` the steps are so small the loss is still 4.30 after 200 epochs — it barely moved. At `lr=10.0` every step overshoots further than the last; the loss explodes to `inf` and then prints `nan`.
</details>

### Exercise 04 · 03 · Remove zero_grad

Remove `optimizer.zero_grad()` and train again; explain in one sentence what happens to the loss and why.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The loss stops falling and swings up and down instead of settling (and with a larger learning rate it blows up), because each step’s gradient is added to all the previous ones — the updates no longer point downhill.
</details>

### Interview-style

Walk through the five-line training loop — what does each line do, and what happens if you remove `zero_grad`?

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`pred = model(x)` is the forward pass: data flows through the model to produce predictions. `loss = loss_fn(pred, y)` measures the error as one number. `optimizer.zero_grad()` clears the gradients left over from the previous step. `loss.backward()` computes the gradient of the loss with respect to every parameter. `optimizer.step()` nudges each parameter a small amount in the direction that reduces the loss.

Without `zero_grad`, gradients accumulate across steps, the updates grow, and the model diverges — the loss rises instead of falling.
</details>